In [1]:
!pip install --quiet kagglehub[pandas-datasets]

import kagglehub
from pathlib import Path

# Descarga completa del dataset
DATASET_DIR = kagglehub.dataset_download("ambityga/imagenet100")

print("Archivos descargados en:", DATASET_DIR)

LABELS_PATH = Path(DATASET_DIR) / "Labels.json"
VAL_DIR = Path(DATASET_DIR) / "val.X"

Using Colab cache for faster access to the 'imagenet100' dataset.
Archivos descargados en: /kaggle/input/imagenet100


In [2]:
# @title
import torch
import torch.nn.functional as F
from pathlib import Path
import json
from PIL import Image
from torchvision.models import resnet34, ResNet34_Weights
import os
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
import matplotlib.pyplot as plt
import numpy as np
from torchvision.models import resnet50, ResNet50_Weights

class ImageNet100ValDataset(Dataset):
    def __init__(self, root_dir, transform=None, labels_json=LABELS_PATH):
        self.root_dir = root_dir
        self.transform = transform

        # Cargar mapeo global desde Labels.json
        with open(labels_json) as f:
            labels = json.load(f)

        # Usar el orden y mapeo original de índices
        self.class_to_idx = {wnid: i for i, wnid in enumerate(labels.keys())}

        # Cargar las muestras
        self.samples = []
        for wnid in self.class_to_idx:
            class_dir = os.path.join(root_dir, wnid)
            if not os.path.isdir(class_dir):
                continue
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(class_dir, f), self.class_to_idx[wnid]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

MEAN_DATASET = [0.485, 0.456, 0.406]
STD_DATASET  = [0.229, 0.224, 0.225]
transform = transforms.Compose([
    transforms.Resize(256),                  # Redimensiona el lado más corto a 256 px
    transforms.CenterCrop(224),              # Recorta el centro a 224×224 (tamaño típico de ImageNet)
    transforms.ToTensor(),                   # Convierte a tensor (0–1)
    transforms.Normalize(                    # Normaliza con medias y desv. estándar de ImageNet
        mean= MEAN_DATASET,
        std =STD_DATASET
    )
])


with open(LABELS_PATH) as f:
    labels = json.load(f)

selected_classes = list(labels.keys())

weights = ResNet34_Weights.DEFAULT

imagenet_classes = weights.meta["categories"]

# Mapear WNID a nombre de clase entendible por el modelo
wnid_to_name = {wnid: labels[wnid].split(',')[0] for wnid in selected_classes}

# Obtener índices dentro de las 1000 clases del modelo
selected_indices_in_model = [imagenet_classes.index(wnid_to_name[wnid]) for wnid in selected_classes]


# ------------------ Estadísticas de ImageNet ------------------
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]


# ------------------ utilidades ------------------
def denormalize_tensor(tensor_norm, mean=MEAN, std=STD):
    """tensor_norm: normalizado -> pixel-space [0,1]"""
    m = torch.tensor(mean, device=tensor_norm.device).view(1, -1, 1, 1)
    s = torch.tensor(std,  device=tensor_norm.device).view(1, -1, 1, 1)
    return (tensor_norm * s + m).clamp(0.0, 1.0)

def normalize_tensor(px, mean=MEAN, std=STD):
    m = torch.tensor(mean, device=px.device).view(1, -1, 1, 1)
    s = torch.tensor(std,  device=px.device).view(1, -1, 1, 1)
    return (px - m) / s

def evaluar_imagen(model, img, selected_indices_in_model):
    """
    Evalúa una sola imagen en el modelo.

    Parámetros:
        model: modelo preentrenado (por ejemplo, resnet50)
        img: tensor de imagen (C, H, W) ya transformado
        selected_indices_in_model: lista de índices de las clases que se quieren evaluar en el modelo

    Devuelve:
        pred_wnid: WNID predicho
        nombre_legible: nombre de la clase predicha
        prob: probabilidad asociada
    """

    input_tensor = img.unsqueeze(0)

    # Inferencia sin gradientes
    with torch.no_grad():
        output = model(input_tensor)

        # Filtrar los logits solo para las clases seleccionadas
        filtered_logits = output[0][selected_indices_in_model]
        filtered_probs = torch.nn.functional.softmax(filtered_logits, dim=0)

        # Elegir la clase más probable
        pred_idx_in_filtered = filtered_probs.argmax().item()
        pred_wnid = selected_classes[pred_idx_in_filtered]
        prob = filtered_probs[pred_idx_in_filtered].item()

        nombre_legible = labels[pred_wnid]

    return pred_wnid, nombre_legible, prob

def load_resnet34(device=None):
    """
    Carga un modelo ResNet34 preentrenado con pesos de ImageNet,
    listo para evaluación, junto con su lista de clases.
    """
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    weights = ResNet34_Weights.DEFAULT
    model = resnet34(weights=weights).to(device).eval()
    imagenet_classes = weights.meta["categories"]
    preprocess = weights.transforms()
    return model, imagenet_classes, preprocess


# =====================================================
# MAPEO ENTRE WNID Y CLASES DEL MODELO
# =====================================================

def load_wnid_mapping(labels_json_path=LABELS_PATH):
    """Carga el mapeo WNID → nombre legible desde un JSON."""
    with open(labels_json_path, "r") as f:
        wnid2name = json.load(f)
    return wnid2name


def build_class_mapping(dataset_root, imagenet_classes, wnid2name):
    """
    Construye un mapeo robusto entre las carpetas locales de ImageNet100
    y los índices de clase del modelo ResNet34 (1000 clases).

    Retorna:
        wnid_to_model_idx, model_idx_to_wnid, selected_indices_in_model
    """
    dataset_root = Path(dataset_root)
    selected_classes = sorted([p.name for p in dataset_root.iterdir() if p.is_dir()])

    wnid_to_model_idx = {}
    for wnid in selected_classes:
        if wnid not in wnid2name:
            continue
        wnid_name = wnid2name[wnid].split(",")[0].lower().strip()
        match = [i for i, c in enumerate(imagenet_classes) if wnid_name in c.lower()]
        if match:
            wnid_to_model_idx[wnid] = match[0]

    selected_indices_in_model = list(wnid_to_model_idx.values())
    model_idx_to_wnid = {v: k for k, v in wnid_to_model_idx.items()}

    print(f"Total carpetas detectadas en {dataset_root}: {len(selected_classes)}")
    print(f"Total clases mapeadas correctamente: {len(selected_indices_in_model)}")

    if not selected_indices_in_model:
        raise ValueError("No se encontró ninguna clase del dataset en las 1000 del modelo. Verifica Labels.json.")

    return wnid_to_model_idx, model_idx_to_wnid, selected_indices_in_model


# =====================================================
# EVALUACIÓN TOP-5 LIMITADA A 100 CLASES
# =====================================================

def top5_for_image_path(model, img_path, preprocess, imagenet_classes,
                        selected_indices_in_model, model_idx_to_wnid, wnid2name,
                        device=None):
    """
    Evalúa una imagen (ruta) y retorna las top-5 predicciones restringidas
    a las clases seleccionadas del dataset (p.ej. las 100 de ImageNet100).

    Devuelve una lista de dicts con:
    - model_idx
    - class_name
    - prob
    - wnid
    - wnid_name
    """
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    img = Image.open(img_path).convert("RGB")
    x = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)[0]
        logits_filtered = logits[selected_indices_in_model]
        probs = F.softmax(logits_filtered, dim=0)

    k = min(5, len(selected_indices_in_model))
    topk = torch.topk(probs, k=k)

    results = []
    for idx_f, p in zip(topk.indices.cpu().tolist(), topk.values.cpu().tolist()):
        model_idx = selected_indices_in_model[idx_f]
        wnid = model_idx_to_wnid.get(model_idx, "unknown")
        wnid_name = wnid2name.get(wnid, "desconocido")
        class_name = imagenet_classes[model_idx]
        results.append({
            "model_idx": model_idx,
            "class_name": class_name,
            "prob": p,
            "wnid": wnid,
            "wnid_name": wnid_name
        })
    return results


def wnid_to_model_index(wnid, weights, labels_json=LABELS_PATH):
    """
    Devuelve el índice (0..999) en weights.meta['categories'] correspondiente al wnid.
    Intenta coincidencia exacta con el nombre "primera parte" del Labels.json,
    y si no encuentra intenta coincidencia parcial.
    Lanza ValueError si no puede mapear.
    """
    with open(labels_json, "r") as f:
        wnid2name = json.load(f)

    if wnid not in wnid2name:
        raise ValueError(f"WNID {wnid} no está en {labels_json}")

    target_name = wnid2name[wnid].split(",")[0].strip().lower()  # p.ej. "wombat"
    imagenet_classes = weights.meta["categories"]

    # 1) buscar coincidencia exacta (comparando nombres lower)
    for i, cname in enumerate(imagenet_classes):
        if cname.lower().strip() == target_name:
            return i

    # 2) intentar coincidencia parcial por tokens
    target_token = target_name.split()[0]
    for i, cname in enumerate(imagenet_classes):
        if target_token in cname.lower():
            return i

    raise ValueError(f"No pude mapear {wnid} -> índice en weights.meta['categories'] (buscado '{target_name}').")

def global_evaluate(model, metodo, param, VAL_DIR="val.X"):

    model.eval()
    val_dataset = ImageNet100ValDataset(VAL_DIR, transform=transform, labels_json=LABELS_PATH)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    correct_top1 = 0
    correct_top5 = 0
    total = 0

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for imgs, labels_idx in val_loader:

        imgs = imgs.to(device)
        labels_idx = labels_idx.to(device)

        if metodo is None:
            att_imgs = imgs
        else:
            att_imgs = torch.stack([
                metodo(model, imgs[i].to(device), labels_idx[i].to(device), param)
                for i in range(len(imgs))
            ]).to(device)

        with torch.no_grad():
            outputs = model(att_imgs)

            filtered_logits = outputs[:, selected_indices_in_model]
            filtered_probs = F.softmax(filtered_logits, dim=1)

            preds_in_filtered = filtered_probs.argmax(dim=1)
            top5_preds = torch.topk(filtered_probs, 5, dim=1).indices

            pred_wnids = [selected_classes[i] for i in preds_in_filtered]
            true_wnids = [list(val_dataset.class_to_idx.keys())[i] for i in labels_idx]


            for i in range(labels_idx.size(0)):
                if labels_idx[i].item() in top5_preds[i]:
                    correct_top5 += 1

            correct_top1 += sum(p == t for p, t in zip(pred_wnids, true_wnids))
            total += len(imgs)

    return correct_top1/total, correct_top5/total


def global_transfer(model_atk, model_trans, metodo, param, VAL_DIR="val.X"):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_atk.to(device).eval()
    model_trans.to(device).eval()
    val_dataset = ImageNet100ValDataset(VAL_DIR, transform=transform, labels_json=LABELS_PATH)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    correct_top1 = 0
    correct_top5 = 0
    total = 0


    for imgs, labels_idx in val_loader:

        imgs = imgs.to(device)
        labels_idx = labels_idx.to(device)

        if metodo is None:
            att_imgs = imgs
        else:
            att_imgs = torch.stack([
                metodo(model_atk, imgs[i].to(device), labels_idx[i].to(device), param)

                for i in range(len(imgs))
            ]).to(device)

        with torch.no_grad():
            outputs = model_trans(att_imgs)


            filtered_logits = outputs[:, selected_indices_in_model]
            filtered_probs = F.softmax(filtered_logits, dim=1)

            preds_in_filtered = filtered_probs.argmax(dim=1)
            top5_preds = torch.topk(filtered_probs, 5, dim=1).indices

            pred_wnids = [selected_classes[i] for i in preds_in_filtered]
            true_wnids = [list(val_dataset.class_to_idx.keys())[i] for i in labels_idx]


            for i in range(labels_idx.size(0)):
                if labels_idx[i].item() in top5_preds[i]:
                    correct_top5 += 1

            correct_top1 += sum(p == t for p, t in zip(pred_wnids, true_wnids))
            total += len(imgs)

    return correct_top1/total, correct_top5/total



In [3]:
# @title
# ------------------ Estadísticas de ImageNet ------------------
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]


# (opcional) image_gradient queda aquí si lo necesitas para otros ataques
def image_gradient(model, img_norm, label):
    model = model.to(device).eval()
    x = img_norm.unsqueeze(0).to(device)
    x.requires_grad_(True)
    y = torch.tensor([label], device=device)
    out = model(x)
    loss = F.cross_entropy(out, y)
    loss.backward()
    return x.grad.detach().squeeze(0)   # grad NORMALIZADO

# ===========================================================
# ✔️ Ataque: ruido Gaussiano en espacio normalizado (misma firma que rFGSM)
# ===========================================================
def generar_imagen_gaussian(model, img_norm, label, args):
    """
    args[0] = sigma (std del ruido Gaussiano) - en espacio NORMALIZADO
    img_norm: imagen NORMALIZADA (C,H,W)
    Devuelve adv_norm: imagen adversarial NORMALIZADA (img_norm + ruido gaussiano)
    """
    sigma = args[0]

    # ruido gaussiano en espacio normalizado
    noise = torch.randn_like(img_norm) * sigma

    adv_norm = img_norm + noise

    # clamp en espacio normalizado (evita valores extremos)
    adv_norm = adv_norm.clamp(-3, 3)

    return adv_norm

In [4]:
# @title
def generar_imagen_rfgsm(model, img_norm, label, args):
    """
    args[0] = eps
    args[1] 0 alpha
    img_norm: imagen NORMALIZADA (C,H,W)
    Devuelve adv_norm: imagen adversarial NORMALIZADA
    """
    noise = torch.empty_like(img_norm).uniform_(-args[1], args[1])
    grad = image_gradient(model, img_norm + noise, label)

    # FGSM en espacio NORMALIZADO
    adv_norm = img_norm + noise - args[0] * grad.sign()

    # Clamp en normalizado evita explosiones
    adv_norm = adv_norm.clamp(-3, 3)

    return adv_norm

In [5]:
# @title
def generar_imagen_PGD(model, img_norm, label, args):
    """
    args[0] = eps      ruido uniforme
    args[1] = alpha    tamaño del paso
    args[2] = iteraciones
    """
    noise = torch.empty_like(img_norm).uniform_(-args[1], args[1])
    img = img_norm + noise
    for _ in range(args[2]):
        grad = image_gradient(model, img, label)
        img = img - args[1] * grad.sign()
        perturb = torch.clamp(img - img_norm, min=-args[0], max=args[0])
        img = img_norm + perturb

    return img

def generar_imagen_RPGD(model, img_norm, label, args):
    """
    args[0] = eps      ruido uniforme
    args[1] = alpha    tamaño del paso
    args[2] = iteraciones
    args[3] = sigma
    """
    noise = torch.empty_like(img_norm).uniform_(-args[1], args[1])
    img = img_norm + noise
    for _ in range(args[2]):
        grad = image_gradient(model, img , label)
        img = img - args[1] * grad.sign()
        perturb = torch.clamp(img - img_norm, min=-args[0], max=args[0])
        img = img_norm + perturb
        noise = torch.empty_like(img_norm).uniform_(-args[3], args[3])
        img = img + noise

    return img
# -------------------------

In [6]:
# @title
# ===========================================================
#  FGSM CORRECTO (opera en ESPACIO NORMALIZADO)
# ===========================================================
def generar_imagen_fgsm(model, img_norm, label, args):
    """
    img_norm: imagen NORMALIZADA (C,H,W)
    Devuelve adv_norm: imagen adversarial NORMALIZADA
    """
    grad = image_gradient(model, img_norm, label)
    eps = torch.tensor(args[0], device=device)
    # FGSM en espacio NORMALIZADO
    adv_norm = img_norm - eps * grad.sign()

    # Clamp en normalizado evita explosiones
    adv_norm = adv_norm.clamp(-3, 3)

    return adv_norm

In [7]:
# @title
from google.colab import files
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = ResNet34_Weights.DEFAULT
model = resnet34(weights=weights).to(device)

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:01<00:00, 61.2MB/s]


In [8]:
dataset = ImageNet100ValDataset(VAL_DIR, transform=transform)

def reducebit(x, bits=6):
    x = x.clone()

    # Asegurar rango [0,1]
    x = x.clamp(0, 1)

    levels = 2 ** bits
    x = torch.round(x * (levels - 1)) / (levels - 1)

    # Volver a limitar por si hay acumulación de errores
    return x.clamp(0, 1)

transform_before_model = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [9]:
# @title
epsilons = 0.05
alpha = 0.01
k = 10
sigma = 0.01
parametros = [epsilons, alpha, k, sigma]

In [10]:

bits = {8:0,
        7: 0,
        6:0,
        5:0,
        4:0,
        3:0,
        2:0,
        1:0
        }

for i in range(len(dataset)):
    # 1. Cargar imagen y etiqueta
    x, y = dataset[i]
    x = x.unsqueeze(0).to(device)  # batch = 1
    y = torch.tensor([y]).to(device)

    # ---- Ataque FGSM ----
    adv = generar_imagen_fgsm(model, x.squeeze(0), y.item(), [epsilons])
    if adv.ndim == 3:
        adv = adv.unsqueeze(0)

    adv_denorm = denormalize_tensor(adv.squeeze(0))
    adv_denorm = adv_denorm

    # ---- DEFENSAS ----
    for bit in bits.keys():
      # Bits 7
      x7 = (reducebit(adv_denorm, bits=bit))
      pred_def7 = model(x7)[:, selected_indices_in_model].argmax(dim=1)
      bits[bit] += (pred_def7 == y).item()
    print(f"{i}/{len(dataset)}")

0/5000
1/5000
2/5000
3/5000
4/5000
5/5000
6/5000
7/5000
8/5000
9/5000
10/5000
11/5000
12/5000
13/5000
14/5000
15/5000
16/5000
17/5000
18/5000
19/5000
20/5000
21/5000
22/5000
23/5000
24/5000
25/5000
26/5000
27/5000
28/5000
29/5000
30/5000
31/5000
32/5000
33/5000
34/5000
35/5000
36/5000
37/5000
38/5000
39/5000
40/5000
41/5000
42/5000
43/5000
44/5000
45/5000
46/5000
47/5000
48/5000
49/5000
50/5000
51/5000
52/5000
53/5000
54/5000
55/5000
56/5000
57/5000
58/5000
59/5000
60/5000
61/5000
62/5000
63/5000
64/5000
65/5000
66/5000
67/5000
68/5000
69/5000
70/5000
71/5000
72/5000
73/5000
74/5000
75/5000
76/5000
77/5000
78/5000
79/5000
80/5000
81/5000
82/5000
83/5000
84/5000
85/5000
86/5000
87/5000
88/5000
89/5000
90/5000
91/5000
92/5000
93/5000
94/5000
95/5000
96/5000
97/5000
98/5000
99/5000
100/5000
101/5000
102/5000
103/5000
104/5000
105/5000
106/5000
107/5000
108/5000
109/5000
110/5000
111/5000
112/5000
113/5000
114/5000
115/5000
116/5000
117/5000
118/5000
119/5000
120/5000
121/5000
122/5000
123

In [11]:
for bit in bits.keys():
  print(f"Defensa {bit}: {bits[bit] /5000*100:.4f} %")

Defensa 8: 42.7200 %
Defensa 7: 43.2800 %
Defensa 6: 44.0000 %
Defensa 5: 44.6000 %
Defensa 4: 45.9400 %
Defensa 3: 48.8600 %
Defensa 2: 34.2800 %
Defensa 1: 18.1200 %


In [12]:

def desempeñobitdefense(ataque, parametros):
  for i in range(len(dataset)):
      x, y = dataset[i]
      x = x.unsqueeze(0).to(device)  # batch = 1
      y = torch.tensor([y]).to(device)
      adv = ataque(model, x.squeeze(0), y.item(), parametros)
      if adv.ndim == 3:
          adv = adv.unsqueeze(0)

      adv_denorm = denormalize_tensor(adv.squeeze(0))
      adv_denorm = adv_denorm

      #  DEFENSAS
      for bit in bits.keys():
        # Bits 7
        x7 = (reducebit(adv_denorm, bits=bit))
        pred_def7 = model(x7)[:, selected_indices_in_model].argmax(dim=1)
        bits[bit] += (pred_def7 == y).item()


In [13]:

bits = {8:0,
        7: 0,
        6:0,
        5:0,
        4:0,
        3:0,
        2:0,
        1:0
        }
desempeñobitdefense(generar_imagen_gaussian, parametros)
for bit in bits.keys():
  print(f"Defensa {bit}: {bits[bit] /5000:.4f}")

Defensa 8: 0.6970
Defensa 7: 0.6966
Defensa 6: 0.6970
Defensa 5: 0.6966
Defensa 4: 0.6922
Defensa 3: 0.6558
Defensa 2: 0.4250
Defensa 1: 0.2012


In [14]:

bits = {8:0,
        7: 0,
        6:0,
        5:0,
        4:0,
        3:0,
        2:0,
        1:0
        }
desempeñobitdefense(generar_imagen_rfgsm, parametros)
for bit in bits.keys():
  print(f"Defensa {bit}: {bits[bit] /5000:.4f}")

Defensa 8: 0.4414
Defensa 7: 0.4384
Defensa 6: 0.4408
Defensa 5: 0.4498
Defensa 4: 0.4706
Defensa 3: 0.4868
Defensa 2: 0.3490
Defensa 1: 0.1844


In [15]:

bits = {8:0,
        7: 0,
        6:0,
        5:0,
        4:0,
        3:0,
        2:0,
        1:0
        }
desempeñobitdefense(generar_imagen_PGD, parametros)
for bit in bits.keys():
  print(f"Defensa {bit}: {bits[bit] /5000:.4f}")

Defensa 8: 0.5632
Defensa 7: 0.5656
Defensa 6: 0.5684
Defensa 5: 0.5758
Defensa 4: 0.5956
Defensa 3: 0.5756
Defensa 2: 0.3646
Defensa 1: 0.1864


In [16]:

bits = {8:0,
        7: 0,
        6:0,
        5:0,
        4:0,
        3:0,
        2:0,
        1:0
        }
desempeñobitdefense(generar_imagen_RPGD, parametros)
for bit in bits.keys():
  print(f"Defensa {bit}: {bits[bit] /5000:.4f}")

Defensa 8: 0.5768
Defensa 7: 0.5758
Defensa 6: 0.5786
Defensa 5: 0.5878
Defensa 4: 0.5996
Defensa 3: 0.5730
Defensa 2: 0.3686
Defensa 1: 0.1832
